# Ashlar's dressed model in five minutes

`ashlar.dressed()` predicts a whole probability distribution for each row of a table, not just a number.
It is the model listed on the [ScoringBench leaderboard](https://scoringbench.com), used here with its default settings.

This notebook fits it on 1,000 California census blocks, looks at a few predicted distributions,
and scores them on 500 held-out blocks beside CatBoost. A GPU runtime makes the fit faster but is not needed.

In [ ]:
!pip -q install "ashlar-tabular[dataframes]" catboost

## 1. Data

The target is a block's median house value in units of $100,000. The census capped it at 5.0,
so about one block in twenty sits exactly at the cap. A single bell curve cannot describe that.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing(as_frame=True)
order = np.random.default_rng(0).permutation(len(data.target))
train, test = order[:1000], order[1000:1500]
X_train, y_train = data.data.iloc[train], data.target.to_numpy()[train]
X_test, y_test = data.data.iloc[test], data.target.to_numpy()[test]
X_train.head()

## 2. Fit

No settings. The fit trains five small networks and five label readers, then joins them into one density.

In [ ]:
import time
import ashlar

start = time.time()
model = ashlar.dressed(random_state=0).fit(X_train, y_train)
print(f"ashlar {ashlar.__version__}, fitted in {time.time() - start:.0f} seconds")

## 3. Look at a few predicted distributions

Each curve is the model's density for one held-out block. The dashed line is the value that block really had.

In [ ]:
dist = model.predict_dist(X_test)
grid = np.linspace(0.0, 5.5, 400)
density = np.exp(dist.log_prob(np.broadcast_to(grid[:, None], (len(grid), len(y_test)))))

rows = [int(np.argsort(y_test)[i]) for i in (25, 250, 475)]
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=False)
for ax, row in zip(axes, rows):
    ax.plot(grid, density[:, row], color="#147a72")
    ax.fill_between(grid, density[:, row], color="#147a72", alpha=0.15)
    ax.axvline(y_test[row], color="black", linestyle="--", linewidth=1)
    ax.set_title(f"Block {row}: actual value {y_test[row]:.2f}")
    ax.set_xlabel("Median house value, $100,000")
axes[0].set_ylabel("Predicted density")
plt.tight_layout()
plt.show()

## 4. Ask the distribution questions

Quantiles, an 80% interval, and the chance that a block is worth more than $300,000.

In [ ]:
import pandas as pd

quantiles = model.predict_quantiles(X_test, [0.1, 0.5, 0.9])
above_3 = 1.0 - dist.cdf(np.full(len(y_test), 3.0))
pd.DataFrame({
    "actual": y_test[rows],
    "mean": model.predict(X_test)[rows],
    "10%": quantiles[rows, 0], "median": quantiles[rows, 1], "90%": quantiles[rows, 2],
    "P(value > 3.0)": above_3[rows],
}, index=[f"block {r}" for r in rows]).round(3)

## 5. Score it on the held-out blocks, beside CatBoost

CatBoost's `RMSEWithUncertainty` predicts a mean and a variance for each row, which is a normal distribution.
Three scores, all on the 500 test blocks:

- **CRPS** scores the whole distribution in the target's units. Lower is better.
- **NLL** is the average negative log density at the actual value. Lower is better.
- **80% coverage** is the share of blocks inside the 10% to 90% interval. It should be near 80%.

In [ ]:
from catboost import CatBoostRegressor
from scipy.stats import norm
from sklearn.model_selection import train_test_split

fit_X, val_X, fit_y, val_y = train_test_split(X_train, y_train, test_size=0.2, random_state=0)
cat = CatBoostRegressor(loss_function="RMSEWithUncertainty", iterations=4000, learning_rate=0.05,
                        od_type="Iter", od_wait=100, random_seed=0, verbose=False)
cat.fit(fit_X, fit_y, eval_set=(val_X, val_y))
mu, var = cat.predict(X_test).T
sd = np.sqrt(var)
z = (y_test - mu) / sd

scores = pd.DataFrame({
    "CRPS": [dist.crps(y_test).mean(),
             (sd * (z * (2 * norm.cdf(z) - 1) + 2 * norm.pdf(z) - 1 / np.sqrt(np.pi))).mean()],
    "NLL": [-dist.log_prob(y_test).mean(), -norm.logpdf(y_test, mu, sd).mean()],
    "80% coverage": [np.mean((y_test >= quantiles[:, 0]) & (y_test <= quantiles[:, 2])),
                     np.mean(np.abs(z) <= norm.ppf(0.9))],
    "RMSE of the mean": [np.sqrt(np.mean((model.predict(X_test) - y_test) ** 2)), np.sqrt(np.mean((mu - y_test) ** 2))],
}, index=["Ashlar dressed", "CatBoost, RMSEWithUncertainty"])
scores.round(3)

## 6. Use your own data

Any table with one numeric target works. Text and category columns can stay as they are.
The dressed model suits a few hundred to a few thousand rows; the fit and the predictions get slow beyond that.

```python
import pandas as pd, ashlar

frame = pd.read_csv("your_file.csv")
y = frame.pop("your_target").to_numpy()
model = ashlar.dressed().fit(frame, y)

dist = model.predict_dist(frame.head(10))
model.predict_quantiles(frame.head(10), [0.05, 0.5, 0.95])
```

More: the [guide to the dressed model](https://github.com/maxmbay/ashlar/blob/main/docs/dressed.md)
and its [ScoringBench results](https://github.com/maxmbay/ashlar/tree/main/benchmarks/scoringbench).